In [1]:
import os, csv, random, pathlib
import numpy as np, pandas as pd
import cv2

# Use the standalone keras package (compatible with the installed version)
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import GlobalAveragePooling2D, Dense
from keras.utils import to_categorical
from keras import optimizers
from keras.callbacks import ModelCheckpoint
from sklearn.model_selection import train_test_split

seed = 42
random.seed(seed)
np.random.seed(seed)


def first_path_matching(pattern):
    """Return the first path matching pattern; works for both files and directories."""
    for p in pathlib.Path(".").rglob(pattern):
        return str(p)
    raise FileNotFoundError(f"No path matches {pattern}")


# Locate CSV and image folders
train_csv_path = first_path_matching("train.csv")
# the image directories are sibling folders named 'train' and 'test'
train_img_dir = first_path_matching("train")  # directory containing training jpgs
test_img_dir = first_path_matching("test")  # directory containing test jpgs

df_labels = pd.read_csv(train_csv_path)  # columns: id, has_cactus
label_dict = dict(zip(df_labels["id"], df_labels["has_cactus"]))

train_files = sorted(
    [f for f in os.listdir(train_img_dir) if f.lower().endswith(".jpg")]
)
X = []
y = []
for fname in train_files:
    img_path = os.path.join(train_img_dir, fname)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    X.append(img)
    y.append(label_dict[fname])
X = np.stack(X)  # (N, 32, 32, 3)
y_array = np.array(y)  # integer labels 0/1
y_onehot = to_categorical(y_array, num_classes=2)  # one‑hot for training



2026-05-05 03:25:33.075965: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777951533.089085      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777951533.093170      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 03:25:33.110231: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y_onehot, test_size=0.2, random_state=seed, stratify=y_array
)



In [3]:
base_model = VGG16(include_top=False, weights="imagenet", input_shape=(32, 32, 3))
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(1024, activation="relu")(x)
pred = Dense(2, activation="softmax")(x)
model = Model(inputs=base_model.input, outputs=pred)

model.compile(
    optimizer=optimizers.SGD(learning_rate=0.0001, momentum=0.9),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)



2026-05-05 03:25:44.816593: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1777951544.817039      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:c5:00.0, compute capability: 8.6


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [4]:
os.makedirs("./output", exist_ok=True)
ckpt_path = "./output/best_model.keras"
checkpoint = ModelCheckpoint(
    ckpt_path, monitor="val_loss", save_best_only=True, mode="min", verbose=0
)

model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=5,
    batch_size=32,
    callbacks=[checkpoint],
    verbose=2,
)

if os.path.exists(ckpt_path):
    model.load_weights(ckpt_path)



Epoch 1/5


I0000 00:00:1777951548.451748     109 service.cc:148] XLA service 0x7f11b80043f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777951548.451781     109 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-05 03:25:48.475722: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1777951548.621076     109 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-05 03:25:49.129879: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_590', 100 bytes spill stores, 100 bytes spill loads

2026-05-05 03:25:49.237464: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_590', 444 bytes spill

355/355 - 15s - 41ms/step - accuracy: 0.9782 - loss: 0.1043 - val_accuracy: 0.9841 - val_loss: 0.0453
Epoch 2/5
355/355 - 4s - 11ms/step - accuracy: 0.9930 - loss: 0.0223 - val_accuracy: 0.9937 - val_loss: 0.0164
Epoch 3/5
355/355 - 4s - 11ms/step - accuracy: 0.9960 - loss: 0.0125 - val_accuracy: 0.9940 - val_loss: 0.0140
Epoch 4/5
355/355 - 3s - 10ms/step - accuracy: 0.9981 - loss: 0.0069 - val_accuracy: 0.9933 - val_loss: 0.0153
Epoch 5/5
355/355 - 3s - 9ms/step - accuracy: 0.9984 - loss: 0.0051 - val_accuracy: 0.9954 - val_loss: 0.0151


In [5]:
test_files = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(".jpg")])
preds = []
for fname in test_files:
    img_path = os.path.join(test_img_dir, fname)
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)
    img = np.expand_dims(img, axis=0)  # (1, 32, 32, 3)
    prob = model.predict(img, verbose=0)[0][1]  # probability for class 1
    preds.append([fname, prob])



In [6]:
submission_path = "./submission.csv"
with open(submission_path, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "has_cactus"])
    writer.writerows(preds)

print(f"Submission written to {submission_path} with {len(preds)} rows.")

Submission written to ./submission.csv with 3325 rows.
